# 一、Pandas的Merge操作
## 1.1 基础参数
- left: 左侧的DataFrame。

- right: 右侧的DataFrame。

- how: 决定了合并的方式，可以是以下四种之一：

  - 'inner': 只保留两个DataFrame中有匹配项的行（交集）。相当于笛卡尔积
  - 'outer': 保留两个DataFrame的所有行，对于缺失的值填充NaN。
  - 'left': 以左侧DataFrame为主，保留其所有行，右侧无匹配时填充NaN。
  - 'right': 以右侧DataFrame为主，保留其所有行，左侧无匹配时填充NaN。
- on: 指定用于连接的列名（必须在左右两边DataFrame中都存在）。如果左右DataFrame的连接列名相同，只需要提供一个列名即可。

- left_on/right_on: 分别指定左侧和右侧DataFrame用于连接的列名。当连接列名不同时，这两个参数很有用。

- left_index/right_index: 布尔值，默认为False。如果设置为True，则使用左侧或右侧的索引作为连接键。

- suffixes: 元组形式，如('_x', '_y')，用于处理重名列时添加的后缀。

- copy: 默认为True，总是复制数据，默认情况下应保持为默认值以避免潜在的问题。



In [ ]:
import pandas as pd
#1.读取数据
df_ratings = pd.read_csv(r"D:\JupyterProject\study-pandas\file\datas\mov\ratings.dat", 
                         sep = "::", engine = "python", 
                         names = ["UserID", "MovieID", "Rating", "Timestamp"])
df_ratings.head()

In [7]:
df_users = pd.read_csv(r"D:\JupyterProject\study-pandas\file\datas\mov\users.dat", 
                         sep = "::", engine = "python", 
                         names = ["UserID", "Gender", "Age", "Occupation", "Zip-code"])
df_users.head()

,UserID,Gender,Age,Occupation,Zip-code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [10]:
df_movies = pd.read_csv(r"D:\JupyterProject\study-pandas\file\datas\mov\movies.dat", 
                         sep = "::", engine = "python", 
                         encoding='ISO-8859-1',
                         names = ["MovieID", "Title", "Genres"])

df_movies.head()

,MovieID,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [24]:
#ratings评分数据 和 users join
df_ratings_users = pd.merge(df_ratings, df_users, on = "UserID", how = "inner")
df_ratings_users.head()
# print(df_ratings_users["MovieID"].value_counts())

2858    3428
260     2991
1196    2990
1210    2883
480     2672
        ... 
3458       1
2226       1
1815       1
398        1
2909       1
Name: MovieID, Length: 3706, dtype: int64


In [18]:
#ratings_users_movies
df_ratings_users_movies = pd.merge(df_ratings_users, df_movies, on = "MovieID", how = "inner")
print(df_ratings_users_movies.head())

   UserID  MovieID  Rating  Timestamp Gender  Age  Occupation Zip-code  \
0       1     1193       5  978300760      F    1          10    48067   
1       2     1193       5  978298413      M   56          16    70072   
2      12     1193       4  978220179      M   25          12    32793   
3      15     1193       4  978199279      M   25           7    22903   
4      17     1193       5  978158471      M   50           1    95350   

                                    Title Genres  
0  One Flew Over the Cuckoo's Nest (1975)  Drama  
1  One Flew Over the Cuckoo's Nest (1975)  Drama  
2  One Flew Over the Cuckoo's Nest (1975)  Drama  
3  One Flew Over the Cuckoo's Nest (1975)  Drama  
4  One Flew Over the Cuckoo's Nest (1975)  Drama  


In [35]:
## 1.2 一个例子帮助理解不同连接方式的差别
left = pd.DataFrame({
        'key': ['K0', 'K1', 'K2', 'K4'],
        'A': ['A0', 'A1', 'A2', 'A3'], 
        'B': ['B0', 'B1', 'B2', 'B3'],
                    })
right = pd.DataFrame({
        'key': ['K0', 'K1', 'K1', 'K3'],
        'C': ['C0', 'C1', 'C2', 'C3'], 
        'D': ['D0', 'D1', 'D2', 'D3'],
                    })

In [36]:
#inner join
# 如果没有明确指定on参数时，Pandas的merge函数会默认使用两个DataFrame中具有相同名称的列作为连接键。
#对于两个表中，inner会把on指定的列的值相同的行，拼成一行放到新表中
pd.merge(left, right, how = 'inner')

,key,A,B,C,D
0,K0,A0,B0,C0,D0
1,K1,A1,B1,C1,D1
2,K1,A1,B1,C2,D2


In [37]:
#left(right)
#操作解释:相当于对于left表的每一行 去right表中遍历每一行 如果找到了on指定列相同的行 就拼上 ；
#          如果没有找到，默认right中有一个on指定列相同的“Nan行”然后拼成新的一行
pd.merge(left, right, how = 'left')

,key,A,B,C,D
0,K0,A0,B0,C0,D0
1,K1,A1,B1,C1,D1
2,K1,A1,B1,C2,D2
3,K2,A2,B2,NaN,NaN
4,K4,A3,B3,NaN,NaN


In [39]:
#outer join
#操作解释：相当于首先执行left的操作 然后执行right join操作只不过如果这一行已经存在于新表中，那么就不加入新表中
pd.merge(left, right, how = "outer")

,key,A,B,C,D
0,K0,A0,B0,C0,D0
1,K1,A1,B1,C1,D1
2,K1,A1,B1,C2,D2
3,K2,A2,B2,NaN,NaN
4,K4,A3,B3,NaN,NaN
5,K3,NaN,NaN,C3,D3
